In [1]:
!pip install fastapi uvicorn nest_asyncio gradio


from fastapi import FastAPI, Body
import nest_asyncio, uvicorn, threading, gradio as gr, requests

app, codes = FastAPI(), []

@app.get("/")
def home():
    return {"info": "Indian Penal Code"}

@app.post("/ipc")
def add(fapi: dict = Body(...)):
    codes.append(fapi)
    return f"Details: Section {fapi['IPC']}, {fapi['case']}, {fapi['punishment']}"

@app.get("/{i}")
def get(i: int):
    if 0 <= i < len(codes):
        d = codes[i]
        return f"IPC: Section {d['IPC']}, Case: {d['case']}, Punishment: {d['punishment']}"
    return "Invalid index"


nest_asyncio.apply()
threading.Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000)).start()

with gr.Blocks() as demo:

    gr.Markdown("# INDIAN PENAL CODE BOT")


    gr.Button("Show Info").click(
        lambda: requests.get("http://127.0.0.1:8000/").json()["info"],
        outputs=gr.Textbox()
    )


    with gr.Row():
        i = gr.Textbox(label="IPC")
        c = gr.Textbox(label="Case")
        p = gr.Textbox(label="Punishment")

        gr.Button("Add IPC").click(
            lambda ipc, case, pun: requests.post(
                "http://127.0.0.1:8000/ipc",
                json={"IPC": ipc, "case": case, "punishment": pun}
            ).text,
            inputs=[i, c, p],
            outputs=gr.Textbox()
        )


    idx = gr.Number(label="IPC Index")

    gr.Button("Get IPC").click(
        lambda index: requests.get(f"http://127.0.0.1:8000/{int(index)}").text,
        inputs=idx,
        outputs=gr.Textbox()
    )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d2aa5ab6d1ab50a872.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
